In [1]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

/data/ll2531/venv/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
with open("processed_data_pkl/imputed_training_data.pkl", "rb") as f:
    traffic_dic = pickle.load(f)

with open("processed_data_pkl/weather_global_2014_2025.pkl", "rb") as f:
    weather = pickle.load(f)

air_qual = pd.read_csv("online_data/air_qual/air_qual.csv")

In [3]:
air_qual.drop(columns=["aerosol_optical_depth ()", "dust (μg/m³)"], inplace=True)
air_qual.dropna(inplace=True)
air_qual.reset_index(inplace=True, drop=True)

In [4]:
air_qual["time"] = pd.to_datetime(air_qual["time"], errors="coerce")
weather["timestamp"] = pd.to_datetime(weather["timestamp"], errors="coerce")

weather = weather[weather["timestamp"].isin(air_qual["time"])]
weather.reset_index(inplace=True, drop=True)

In [5]:
dataset = pd.concat([weather, air_qual], axis=1)
dataset.drop(columns=["time"], inplace=True)

# setting up the graph

In [6]:
import networkx as nx
import torch
from torch_geometric.utils import to_networkx
from torch_geometric.nn import GCNConv
from torch.nn import Linear
import geopy.distance
from torch_geometric.data import Data


/data/ll2531/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
G = nx.DiGraph()

for sensor_id, directions in traffic_dic.items():
    for direction, df in directions.items():
        origin_lat = df["latitude"].iloc[0]
        origin_lon = df["longitude"].iloc[0]

        G.add_node(sensor_id, pos=(origin_lon, origin_lat))
        
        for sensor_id_2, directions_2 in traffic_dic.items():
            if sensor_id == sensor_id_2:
                continue

            for direction_2, df_2 in directions_2.items():
                target_lat = df_2["latitude"].iloc[0]
                target_lon = df_2["longitude"].iloc[0]
                
                distance_m = geopy.distance.geodesic(
                    (origin_lat, origin_lon), 
                    (target_lat, target_lon)
                ).m

                if distance_m < 1000:
                    G.add_edge(
                        sensor_id, 
                        sensor_id_2, 
                        distance=distance_m, 
                        dir=direction
                    )

In [8]:
manual_edges = [(11,23), (9,22), (9,23), (1,12), (2,14), (3,14),(4,14), (5,15), (8,20), (6,19), (6,16), (8,19), (7,20), (7,19), (6,5), (17,14), (15,14), (16,14)]

In [9]:
import geopy.distance

node_positions = nx.get_node_attributes(G, "pos")

for source, target in manual_edges:
    lon1, lat1 = node_positions[source]
    lon2, lat2 = node_positions[target]

    distance_m = geopy.distance.geodesic((lat1, lon1), (lat2, lon2)).m
    G.add_edge(source, target, distance=distance_m)

In [10]:
sensor_ids = list(G.nodes())
node_map = {sid: idx for idx, sid in enumerate(sensor_ids)}
num_nodes = len(sensor_ids)

sample_df = list(traffic_dic[sensor_ids[0]].values())[0]

exclude_cols = ['latitude', 'longitude', 'timestamp', 'time', 'miles', "avtime", "hour"]
base_features = [col for col in sample_df.columns if col not in exclude_cols]

time_features = ['hour_sin', 'hour_cos', 'day_sin', 'day_cos']
all_features = base_features + time_features

num_timestamps = len(sample_df)
num_features = len(all_features)

x_tensor = torch.zeros((num_nodes, num_features, num_timestamps), dtype=torch.float)

for sensor_id in sensor_ids:
    node_idx = node_map[sensor_id]
    traffic_df = list(traffic_dic[sensor_id].values())[0].copy()
    
    traffic_df['timestamp'] = pd.to_datetime(traffic_df['timestamp'])
    
    traffic_df['hour_sin'] = np.sin(2 * np.pi * traffic_df['timestamp'].dt.hour / 24.0)
    traffic_df['hour_cos'] = np.cos(2 * np.pi * traffic_df['timestamp'].dt.hour / 24.0)
    traffic_df['day_sin'] = np.sin(2 * np.pi * traffic_df['timestamp'].dt.dayofweek / 7.0)
    traffic_df['day_cos'] = np.cos(2 * np.pi * traffic_df['timestamp'].dt.dayofweek / 7.0)
    traffic_df["count_imputed"] = traffic_df["count_imputed"].apply(lambda x: 0 if x == False else 1)
    traffic_features_matrix = traffic_df[all_features].values.T
    x_tensor[node_idx, :, :] = torch.tensor(traffic_features_matrix, dtype=torch.float)

edge_list = []
edge_attr_list = []
for u, v, data in G.edges(data=True):
    edge_list.append([node_map[u], node_map[v]])
    edge_attr_list.append([data['distance']])

edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
edge_attr = torch.tensor(edge_attr_list, dtype=torch.float)

pyg_dataset = Data(x=x_tensor, edge_index=edge_index, edge_attr=edge_attr)

In [11]:
import plotly.graph_objects as go

num_nodes = len(node_map)
coords_traffic = np.zeros((num_nodes, 2))

for sensor_id, node_idx in node_map.items():
    lon, lat = G.nodes[sensor_id]['pos']
    coords_traffic[node_idx, 0] = lon
    coords_traffic[node_idx, 1] = lat

edge_t_to_t = pyg_dataset.edge_index


def visualize_traffic_spatial_graph(coords_traffic, edge_t_to_t, mapbox_style="carto-positron"):
    """
    Visualizes the traffic sensor spatial graph directly on a map using Plotly Scattermapbox.
    """
    fig = go.Figure()

    # --- DRAW TRAFFIC-TO-TRAFFIC EDGES ---
    t_edge_lon, t_edge_lat = [], []
    t_start_nodes = edge_t_to_t[0].numpy()
    t_end_nodes = edge_t_to_t[1].numpy()
    
    for src, dst in zip(t_start_nodes, t_end_nodes):
        # Plotly draws continuous paths; adding None breaks the line between distinct edges
        t_edge_lon.extend([coords_traffic[src, 0], coords_traffic[dst, 0], None])
        t_edge_lat.extend([coords_traffic[src, 1], coords_traffic[dst, 1], None])
        
    fig.add_trace(go.Scattermapbox(
        lon=t_edge_lon, lat=t_edge_lat,
        mode='lines',
        line=dict(width=1.5, color='rgba(50, 150, 250, 0.6)'),
        name='Traffic-to-Traffic Edges',
        hoverinfo='none'
    ))

    fig.add_trace(go.Scattermapbox(
        lon=coords_traffic[:, 0], lat=coords_traffic[:, 1],
        mode='markers',
        marker=dict(size=10, color='blue', opacity=0.85),
        name='Traffic Sensor Nodes',
        text=[f"Node Index: {i}<br>Sensor ID: {sensor_ids[i]}" for i in range(len(coords_traffic))],
        hoverinfo='text'
    ))

    center_lat = np.mean(coords_traffic[:, 1])
    center_lon = np.mean(coords_traffic[:, 0])

    fig.update_layout(
        title=dict(text='Spatio-Temporal Traffic Graph Topology', font=dict(size=18)),
        autosize=True,
        hovermode='closest',
        showlegend=True,
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor="rgba(255,255,255,0.7)"),
        mapbox=dict(
            style=mapbox_style,
            bearing=0,
            center=dict(lat=center_lat, lon=center_lon),
            pitch=0,
            zoom=12
        ),
        width=1100,
        height=750,
        margin=dict(r=0, t=40, l=0, b=0)
    )
    
    fig.show()

visualize_traffic_spatial_graph(coords_traffic, edge_t_to_t)

/tmp/ipykernel_3572651/3245259071.py:30: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
/tmp/ipykernel_3572651/3245259071.py:38: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler

# ==========================================
# 1. GRAPH STRUCTURE + TEMPORAL FEATURE PREPARATION
# ==========================================
# Extract dimensions from the existing graph tensor
num_nodes_spatial = pyg_dataset.x.shape[0]
num_features_spatial = pyg_dataset.x.shape[1]
total_timestamps = pyg_dataset.x.shape[2]

# Build the spatial adjacency matrix from the PyG edge index
adj_matrix = np.zeros((num_nodes_spatial, num_nodes_spatial), dtype=np.float32)
edges = pyg_dataset.edge_index.numpy()
adj_matrix[edges[0], edges[1]] = 1.0
adj_matrix += np.eye(num_nodes_spatial, dtype=np.float32)

deg = np.sum(adj_matrix, axis=1)
deg_inv_sqrt = np.power(deg, -0.5, where=deg > 0)
deg_inv_sqrt[deg == 0] = 0.0
D_inv_sqrt = np.diag(deg_inv_sqrt)
normalized_adj = D_inv_sqrt @ adj_matrix @ D_inv_sqrt

# Convert traffic tensor to a time-major layout for windowing
traffic_np = pyg_dataset.x.numpy()
traffic_time_major = np.transpose(traffic_np, (2, 0, 1))  # (timestamps, nodes, features)

# Build a multivariate temporal dataframe for the PM2.5 branch
# This mirrors the logic from notebooks 4.5/4.6 while keeping it separate from the graph branch.
temporal_df = pd.merge(weather, air_qual, left_on="timestamp", right_on="time", how="inner")
temporal_df = temporal_df.sort_values("timestamp").set_index("timestamp")
temporal_df = temporal_df.asfreq("h")
temporal_df = temporal_df.drop(columns=["time"], errors="ignore")
temporal_df["hour_sin"] = temporal_df.index.hour.apply(lambda x: np.sin(2 * np.pi * x / 24.0))
temporal_df["hour_cos"] = temporal_df.index.hour.apply(lambda x: np.cos(2 * np.pi * x / 24.0))
temporal_df = temporal_df.dropna()

temporal_feature_cols = [
    "pm2_5 (μg/m³)",
    "hour_sin",
    "hour_cos",
    "temperature_2m (°C)",
    "surface_pressure (hPa)",
    "wind_speed_100m (km/h)",
    "precipitation (mm)",
]
temporal_feature_cols = [col for col in temporal_feature_cols if col in temporal_df.columns]
target_col = "pm2_5 (μg/m³)"

scaler_X_temporal = MinMaxScaler()
scaler_y_temporal = MinMaxScaler()
scaled_temporal = scaler_X_temporal.fit_transform(temporal_df[temporal_feature_cols])
scaled_target = scaler_y_temporal.fit_transform(temporal_df[[target_col]])

# Optional extra branch for weather-only features; keep it disabled by default.
USE_WEATHER_BRANCH = False
if USE_WEATHER_BRANCH:
    weather_feature_cols = [
        "temperature_2m (°C)",
        "surface_pressure (hPa)",
        "wind_speed_100m (km/h)",
        "precipitation (mm)",
    ]
    weather_feature_cols = [col for col in weather_feature_cols if col in temporal_df.columns]
    scaler_X_weather = MinMaxScaler()
    scaled_weather = scaler_X_weather.fit_transform(temporal_df[weather_feature_cols])
else:
    scaled_weather = None

# Sliding-window generation for the two main branches
lookback = 24 * 3
forecast_steps = 24

traffic_timestamps = pd.to_datetime(temporal_df.index[:traffic_time_major.shape[0]])
traffic_series_map = pd.Series(range(len(traffic_timestamps)), index=traffic_timestamps)

x_traffic = []
x_temporal = []
x_weather_branch = []
y = []

for i in range(lookback, len(temporal_df) - forecast_steps + 1):
    current_time_window = temporal_df.index[i - lookback : i]
    try:
        traffic_indices = traffic_series_map.loc[current_time_window].values.astype(int)
        if len(traffic_indices) != lookback or np.isnan(traffic_indices).any():
            continue

        traffic_slice = traffic_time_major[traffic_indices, :, :]
        if traffic_slice.shape != (lookback, num_nodes_spatial, num_features_spatial):
            continue

        x_traffic.append(traffic_slice)
        x_temporal.append(scaled_temporal[i - lookback : i, :])
        if USE_WEATHER_BRANCH:
            x_weather_branch.append(scaled_weather[i - lookback : i, :])
        y.append(scaled_target[i : i + forecast_steps, 0])
    except KeyError:
        continue

x_traffic = np.array(x_traffic, dtype=np.float32)
x_temporal = np.array(x_temporal, dtype=np.float32)
y = np.array(y, dtype=np.float32)

if USE_WEATHER_BRANCH:
    x_weather_branch = np.array(x_weather_branch, dtype=np.float32)

print(f"Traffic branch shape: {x_traffic.shape}")
print(f"Temporal branch shape: {x_temporal.shape}")
print(f"Target shape: {y.shape}")
if USE_WEATHER_BRANCH:
    print(f"Weather branch shape: {x_weather_branch.shape}")

# Train/Test split
split = int(len(y) * 0.7)
x_train_traffic, x_test_traffic = x_traffic[:split], x_traffic[split:]
x_train_temporal, x_test_temporal = x_temporal[:split], x_temporal[split:]
y_train, y_test = y[:split], y[split:]

if USE_WEATHER_BRANCH:
    x_train_weather, x_test_weather = x_weather_branch[:split], x_weather_branch[split:]
else:
    x_train_weather = None
    x_test_weather = None

# Repeat the adjacency for each sample in the dataset
adj_train = np.repeat(np.expand_dims(normalized_adj, axis=0), len(x_train_traffic), axis=0)
adj_test = np.repeat(np.expand_dims(normalized_adj, axis=0), len(x_test_traffic), axis=0)

num_temporal_features = x_temporal.shape[-1]


2026-06-14 16:51:42.402698: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-14 16:51:42.441996: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-14 16:51:43.283767: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [ ]:
# ==========================================
# 2. DATASET ALIGNMENT & SLIDING WINDOW GENERATION
# ==========================================
# Force the 'timestamp' column to be a true datetime index for mapping
if "timestamp" in dataset.columns:
    dataset.set_index("timestamp", inplace=True)
dataset.index = pd.to_datetime(dataset.index)

features_df = dataset.drop(columns=["pm2_5 (μg/m³)"])
target_df = dataset[["pm2_5 (μg/m³)"]]

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
scaled_X = scaler_X.fit_transform(features_df)
scaled_y = scaler_y.fit_transform(target_df)

lookback = 24 * 3     
forecast_steps = 24 

traffic_timestamps = pd.to_datetime(dataset.index[:traffic_time_major.shape[0]])
traffic_series_map = pd.Series(range(len(traffic_timestamps)), index=traffic_timestamps)

x_weather = []
x_traffic = []
y = []

# Safe dynamic calendar sliding window loop
for i in range(lookback, len(dataset) - forecast_steps + 1):
    current_time_window = dataset.index[i - lookback : i]
    
    try:
        # Match the current row dates with the corresponding traffic array positions
        traffic_indices = traffic_series_map.loc[current_time_window].values
        
        # Guard: Check that the window indices are unbroken and have no NaN values
        if len(traffic_indices) != lookback or np.isnan(traffic_indices).any():
            continue
            
        traffic_indices = traffic_indices.astype(int)
        traffic_slice = traffic_time_major[traffic_indices, :, :]
        
        # Guard: Ensure the extracted matrix window shape is perfectly homogeneous
        if traffic_slice.shape != (lookback, num_nodes_spatial, num_features_spatial):
            continue

        # Append synchronized elements if all guards pass
        x_traffic.append(traffic_slice)
        x_weather.append(scaled_X[i - lookback : i, :])
        y.append(scaled_y[i : i + forecast_steps, 0])
        
    except KeyError:
        # Gracefully skip any time blocks completely missing from either feature matrix
        continue

# Secure structural conversion to NumPy matrices
x_weather = np.array(x_weather, dtype=np.float32)
x_traffic = np.array(x_traffic, dtype=np.float32)
y = np.array(y, dtype=np.float32)

# Verify alignment shapes before building graph model
print(f"Weather sequences array shape: {x_weather.shape}")  # (Samples, 120, features)
print(f"Traffic sequences array shape: {x_traffic.shape}")  # (Samples, 120, 24, features)
print(f"Target vector output shape:     {y.shape}")          # (Samples, 24)

# Synchronized Train/Test Partitioning (70% Train, 30% Evaluation)
split = int(len(y) * 0.7)

x_train_weather, x_test_weather = x_weather[:split], x_weather[split:]
x_train_traffic, x_test_traffic = x_traffic[:split], x_traffic[split:]
y_train, y_test = y[:split], y[split:]

# Replicate the static topology matrix along the final dataset split sizes
adj_train = np.repeat(np.expand_dims(normalized_adj, axis=0), len(x_train_weather), axis=0)
adj_test = np.repeat(np.expand_dims(normalized_adj, axis=0), len(x_test_weather), axis=0)

Weather sequences array shape: (6831, 120, 16)
Traffic sequences array shape: (6831, 120, 24, 11)
Target vector output shape:     (6831, 24)


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Dense, Dropout, Layer, Concatenate, Conv2D, MaxPooling2D, Flatten
)

# ==========================================
# 2. DUAL-BRANCH SPATIO-TEMPORAL MODEL
# ==========================================
class GraphConvLayer(Layer):
    def __init__(self, units, **kwargs):
        super(GraphConvLayer, self).__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        traffic_shape = input_shape[0] if isinstance(input_shape, list) else input_shape
        self.w = self.add_weight(
            shape=(traffic_shape[-1], self.units),
            initializer="glorot_uniform",
            trainable=True,
            name="gcn_weight"
        )
        super(GraphConvLayer, self).build(input_shape)

    def call(self, inputs, adj):
        transformed = tf.matmul(inputs, self.w)
        out = tf.einsum('bij,btjc->btic', adj, transformed)
        return tf.nn.relu(out)


# --- Branch 1: Graph-based traffic encoder ---
traffic_inputs = Input(shape=(lookback, num_nodes_spatial, num_features_spatial), name="traffic_input")
adj_inputs = Input(shape=(num_nodes_spatial, num_nodes_spatial), name="adjacency_input")

t_conv1 = Conv2D(filters=32, kernel_size=(3, 1), padding='same', activation='relu')(traffic_inputs)
t_conv1 = Dropout(0.3)(t_conv1)

s_gcn = GraphConvLayer(units=32)(t_conv1, adj_inputs)
s_gcn = Dropout(0.3)(s_gcn)

pool_spatial = MaxPooling2D(pool_size=(2, 1))(s_gcn)
flat_branch_spatial = Flatten()(pool_spatial)

# --- Branch 2: Multivariate temporal PM2.5 encoder ---
temporal_inputs = Input(shape=(lookback, num_temporal_features), name="temporal_input")

temporal_lstm_1 = LSTM(64, return_sequences=True)(temporal_inputs)
temporal_drop_1 = Dropout(0.3)(temporal_lstm_1)

temporal_lstm_2 = LSTM(32, return_sequences=False)(temporal_drop_1)
flat_branch_temporal = Dropout(0.3)(temporal_lstm_2)

# --- Optional Branch 3: weather-only encoder ---
if USE_WEATHER_BRANCH:
    weather_inputs = Input(shape=(lookback, x_train_weather.shape[2]), name="weather_input")
    weather_lstm = LSTM(32, return_sequences=False)(weather_inputs)
    flat_branch_weather = Dropout(0.3)(weather_lstm)
    merged_features = Concatenate()([flat_branch_spatial, flat_branch_temporal, flat_branch_weather])
else:
    merged_features = Concatenate()([flat_branch_spatial, flat_branch_temporal])

# --- Fusion and output head ---
dense_1 = Dense(64, activation="relu")(merged_features)
drop_5 = Dropout(0.4)(dense_1)
outputs = Dense(forecast_steps, name="forecast_output")(drop_5)

# Build and compile
model = Model(inputs=[traffic_inputs, adj_inputs, temporal_inputs] + ([weather_inputs] if USE_WEATHER_BRANCH else []), outputs=outputs)
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

model.summary()


I0000 00:00:1781452321.649577 3572651 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46564 MB memory:  -> device: 0, name: NVIDIA RTX 6000 Ada Generation, pci bus id: 0000:ac:00.0, compute capability: 8.9
I0000 00:00:1781452321.650539 3572651 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 46661 MB memory:  -> device: 1, name: NVIDIA RTX 6000 Ada Generation, pci bus id: 0000:ca:00.0, compute capability: 8.9


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ traffic_input       │ (None, 120, 24,   │          0 │ -                 │
│ (InputLayer)        │ 11)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 120, 24,   │      1,088 │ traffic_input[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 120, 24,   │          0 │ conv2d[0][0]      │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ adjacency_input     │ (None, 24, 24)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ weather_input       │ (None, 120, 16)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ graph_conv_layer    │ (None, 120, 24,   │      1,024 │ dropout[0][0],    │
│ (GraphConvLayer)    │ 32)               │            │ adjacency_input[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 120, 64)   │     20,736 │ weather_input[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 120, 24,   │          0 │ graph_conv_layer… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 120, 64)   │          0 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 60, 24,    │          0 │ dropout_1[0][0]   │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 32)        │     12,416 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 46080)     │          0 │ max_pooling2d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 32)        │          0 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 46112)     │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │  2,951,232 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ forecast_output     │ (None, 24)        │      1,560 │ dropout_4[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,988,056 (11.40 MB)

 Trainable params: 2,988,056 (11.40 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# ==========================================
# 3. TRAINING WITH TWO MAIN INPUTS
# ==========================================
batch_size = 32

# Generator loops that stream slices on demand

def train_data_generator():
    for i in range(len(x_train_traffic)):
        sample = {
            "traffic_input": x_train_traffic[i],
            "adjacency_input": normalized_adj,
            "temporal_input": x_train_temporal[i]
        }
        if USE_WEATHER_BRANCH:
            sample["weather_input"] = x_train_weather[i]
        yield sample, y_train[i]


def test_data_generator():
    for i in range(len(x_test_traffic)):
        sample = {
            "traffic_input": x_test_traffic[i],
            "adjacency_input": normalized_adj,
            "temporal_input": x_test_temporal[i]
        }
        if USE_WEATHER_BRANCH:
            sample["weather_input"] = x_test_weather[i]
        yield sample, y_test[i]

# Define expected input/output shapes for TensorFlow's execution graph
input_signature = []
input_signature.append(tf.TensorSpec(shape=(lookback, num_nodes_spatial, num_features_spatial), dtype=tf.float32))
input_signature.append(tf.TensorSpec(shape=(num_nodes_spatial, num_nodes_spatial), dtype=tf.float32))
input_signature.append(tf.TensorSpec(shape=(lookback, num_temporal_features), dtype=tf.float32))
if USE_WEATHER_BRANCH:
    input_signature.append(tf.TensorSpec(shape=(lookback, x_train_weather.shape[2]), dtype=tf.float32))

output_signature = tf.TensorSpec(shape=(forecast_steps,), dtype=tf.float32)

train_dataset = (
    tf.data.Dataset.from_generator(
        train_data_generator,
        output_signature=(tuple(input_signature), output_signature)
    )
    .shuffle(buffer_size=1024)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    tf.data.Dataset.from_generator(
        test_data_generator,
        output_signature=(tuple(input_signature), output_signature)
    )
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=10
)


Epoch 1/15


E0000 00:00:1781452324.048449 3572651 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2026-06-14 16:52:04.464936: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002


    150/Unknown 4s 13ms/step - loss: 6286.5857 - mae: 17.4878

2026-06-14 16:52:07.234681: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2026-06-14 16:52:07.234732: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[StatefulPartitionedCall/gradient_tape/compile_loss/mse/mod/_55]]
2026-06-14 16:52:07.234793: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 8815296611261983113
2026-06-14 16:52:07.234817: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 9822076635702336267
2026-06-14 16:52:07.234826: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 9664362918010951813
2026-06-14 16:52:07.234836: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelle

150/150 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - loss: 1391.0562 - mae: 3.9949 - val_loss: 0.0155 - val_mae: 0.1049
Epoch 2/15


...

2026-06-14 16:52:38.554625: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 14519700587161489390
2026-06-14 16:52:38.554650: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 1909640095527926180
2026-06-14 16:52:38.554656: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 4104543321450902874


150/150 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 0.0190 - mae: 0.0928 - val_loss: 0.0096 - val_mae: 0.0718
Epoch 14/15
147/150 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0197 - mae: 0.0904

2026-06-14 16:52:43.428299: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 9664362918010951813
2026-06-14 16:52:43.428357: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 2469666225211897432
2026-06-14 16:52:43.428373: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 4124820129507051337
2026-06-14 16:52:43.428387: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 4278646307019381188
2026-06-14 16:52:43.428399: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 2728875619901679044
2026-06-14 16:52:43.428412: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 5029650404267804198
2026-06-14 16:52:43.428433: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv ite

150/150 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 0.0183 - mae: 0.0897 - val_loss: 0.0091 - val_mae: 0.0685
Epoch 15/15
150/150 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - loss: 0.0177 - mae: 0.0868 - val_loss: 0.0086 - val_mae: 0.0655


2026-06-14 16:52:46.715043: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 8318239189047909316
2026-06-14 16:52:46.715099: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 14519700587161489390
2026-06-14 16:52:46.715114: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 1909640095527926180
2026-06-14 16:52:46.715130: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 8869244479079780361
2026-06-14 16:52:46.715141: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 4104543321450902874
2026-06-14 16:52:46.715170: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 3372390267612253866


In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# ==========================================
# 1. GENERATE MODEL PREDICTIONS
# ==========================================
# Pass the optimized validation dataset wrapper directly into predict
pred = model.predict(test_dataset)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("--- Multi-Input STGCN + LSTM Evaluation ---")
print("MAE: ", mae)
print("RMSE:", rmse)
print("R²:  ", r2)

# Continue with your exact original Plotly plotting logic below this line...

65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step
--- Multi-Input STGCN + LSTM Evaluation ---
MAE:  0.0654565766453743
RMSE: 0.09280429126260888
R²:   -0.9152535796165466


In [17]:
# ==========================================
# 2. POST-PROCESSING & SCALE REVERSAL
# ==========================================
y_test_pred_rescaled = scaler_y.inverse_transform(pred)
y_test_true_rescaled = scaler_y.inverse_transform(y_test)

test_start_idx = split + lookback
pm25_raw = air_qual["pm2_5 (μg/m³)"].values

# Set up continuous sequential indices along the X-axis
history_x = list(range(lookback))
forecast_x = list(range(lookback, lookback + forecast_steps))


In [18]:
# ==========================================
# 3. INTERACTIVE PLOTLY DROPDOWN BUILDER
# ==========================================
fig = go.Figure()

# Initialize with the very first test item (Sample 0)
sample_idx = 0
global_start = test_start_idx + sample_idx

fig.add_trace(go.Scatter(
    x=history_x, 
    y=pm25_raw[global_start - lookback : global_start],
    mode='lines+markers', name='Historical PM2.5', line=dict(color='blue')
))

fig.add_trace(go.Scatter(
    x=forecast_x, 
    y=y_test_true_rescaled[sample_idx],
    mode='lines+markers', name='Actual Future', line=dict(color='green', dash='dash')
))

fig.add_trace(go.Scatter(
    x=forecast_x, 
    y=y_test_pred_rescaled[sample_idx],
    mode='lines+markers', name='Predicted Forecast', line=dict(color='red')
))

# Generate dropdown button arrays dynamically
num_samples_to_show = min(50, len(x_test_weather))
buttons = []

for idx in range(num_samples_to_show):
    # Traces toggle visible in sets of 3 (History, Actual, Predicted)
    visibility = [False] * (num_samples_to_show * 3)
    visibility[idx * 3] = True
    visibility[idx * 3 + 1] = True
    visibility[idx * 3 + 2] = True
    
    button = dict(
        label=f"Sample {idx}",
        method="update",
        args=[
            {"visible": visibility},
            {"title": f"PM2.5 Timeline: {lookback}h History + {forecast_steps}h Forecast (Sample {idx})"}
        ]
    )
    buttons.append(button)

# Construct and attach the hidden trace variations (Samples 1 to 50) upfront
for idx in range(1, num_samples_to_show):
    g_start = test_start_idx + idx
    
    # Hidden Historical Context Trace
    fig.add_trace(go.Scatter(
        x=history_x, y=pm25_raw[g_start - lookback : g_start],
        mode='lines+markers', name='Historical PM2.5', line=dict(color='blue'), visible=False
    ))
    # Hidden Ground Truth Forecast Trace
    fig.add_trace(go.Scatter(
        x=forecast_x, y=y_test_true_rescaled[idx],
        mode='lines+markers', name='Actual Future', line=dict(color='green', dash='dash'), visible=False
    ))
    # Hidden Predictive Path Trace
    fig.add_trace(go.Scatter(
        x=forecast_x, y=y_test_pred_rescaled[idx],
        mode='lines+markers', name='Predicted Forecast', line=dict(color='red'), visible=False
    ))

# ==========================================
# 4. CHART LAYOUT SETTINGS
# ==========================================
fig.update_layout(
    updatemenus=[dict(
        active=0, 
        buttons=buttons, 
        direction="down", 
        pad={"r": 10, "t": 10}, 
        showactive=True, 
        x=0.02, xanchor="left", 
        y=1.15, yanchor="top"
    )],
    title=f"PM2.5 Timeline: {lookback}h History + {forecast_steps}h Forecast (Sample 0)",
    xaxis_title="Timeline (Hours)",
    yaxis_title="PM2.5 Concentration (μg/m³)",
    height=600,
    showlegend=True
)

# Position the vertical threshold bar precisely between historical context and forecast start
fig.add_vline(x=lookback - 0.5, line_width=2, line_dash="dash", line_color="gray")

fig.show()